In [1]:
import pandas as pd
import duckdb
import numpy as np

In [2]:
set_types = {12: str, 14: str, 15: str, 18: str, 19: str, 20: str, 21: str}

In [3]:
# Load games with identifier-like columns preserved as text.
games = pd.read_csv("../data/raw/Games.csv", dtype=set_types).reset_index()

In [4]:
games.head()

,index,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,...,gameLabel,gameSubLabel,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials,gameDate
0,0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,90,...,NBA Finals,Game 5,Game 5,18984.0,1000118,Frost Bank Center,San Antonio,TX,"James Capers, Scott Foster, Mitchell Ervin, Ty...",2026-06-13 20:30:00
1,1,42500404,2026-06-10 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,107,...,NBA Finals,Game 4,Game 4,19812.0,30,Madison Square Garden,New York,NY,"Courtney Kirkland, Zach Zarba, James Williams,...",2026-06-10 20:30:00
2,2,42500403,2026-06-08 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,111,...,NBA Finals,Game 3,Game 3,19812.0,30,Madison Square Garden,New York,NY,"Marc Davis, John Goble, Curtis Blair, Nick Buc...",2026-06-08 20:30:00
3,3,42500402,2026-06-05 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,104,...,NBA Finals,Game 2,Game 2,19014.0,1000118,Frost Bank Center,San Antonio,TX,"Tony Brothers, Josh Tiven, Mitchell Ervin, Tyl...",2026-06-05 20:30:00
4,4,42500401,2026-06-03 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,95,...,NBA Finals,Game 1,Game 1,18835.0,1000118,Frost Bank Center,San Antonio,TX,"James Capers, JB DeRosa, Scott Foster, Sean Wr...",2026-06-03 20:30:00


In [5]:
# Separate the source timestamp into date and time fields.
games[["game_Date", "game_Time"]] = games["gameDateTimeEst"].str.split(" ", expand=True)

In [6]:
# Limit games to the historical transaction window.
query = """
    SELECT * FROM games
    WHERE game_date BETWEEN '1976-10-21' AND '2019-04-10'
"""

In [7]:
transaction_timeframe = duckdb.sql(query).df().reset_index().copy()

In [8]:
transaction_timeframe.head()

,level_0,index,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,...,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials,gameDate,game_Date,game_Time
0,0,9485,21801228,2019-04-10 22:30:00,Denver,Nuggets,1610612743,Minnesota,Timberwolves,1610612750,...,None,NaN,139,None,None,None,None,2019-04-10 22:30:00,2019-04-10,22:30:00
1,1,9486,21801229,2019-04-10 22:30:00,Los Angeles,Clippers,1610612746,Utah,Jazz,1610612762,...,None,NaN,137,None,None,None,None,2019-04-10 22:30:00,2019-04-10,22:30:00
2,2,9487,21801230,2019-04-10 22:30:00,Portland,Trail Blazers,1610612757,Sacramento,Kings,1610612758,...,None,NaN,51,None,None,None,None,2019-04-10 22:30:00,2019-04-10,22:30:00
3,3,9488,21801220,2019-04-10 20:00:00,Atlanta,Hawks,1610612737,Indiana,Pacers,1610612754,...,None,NaN,650,None,None,None,None,2019-04-10 20:00:00,2019-04-10,20:00:00
4,4,9489,21801221,2019-04-10 20:00:00,Brooklyn,Nets,1610612751,Miami,Heat,1610612748,...,None,NaN,461,None,None,None,None,2019-04-10 20:00:00,2019-04-10,20:00:00


In [9]:
transaction_timeframe.tail()

,level_0,index,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,...,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials,gameDate,game_Date,game_Time
51702,51702,61187,27600015,1976-10-22 19:00:00,Washington,Bullets,1610612764,Los Angeles,Lakers,1610612747,...,None,NaN,0,None,None,None,None,1976-10-22 19:00:00,1976-10-22,19:00:00
51703,51703,61188,27600004,1976-10-21 19:00:00,Indiana,Pacers,1610612754,Boston,Celtics,1610612738,...,None,NaN,0,None,None,None,None,1976-10-21 19:00:00,1976-10-21,19:00:00
51704,51704,61189,27600005,1976-10-21 19:00:00,Milwaukee,Bucks,1610612749,Buffalo,Braves,1610612746,...,None,NaN,0,None,None,None,None,1976-10-21 19:00:00,1976-10-21,19:00:00
51705,51705,61190,27600006,1976-10-21 19:00:00,Atlanta,Hawks,1610612737,Houston,Rockets,1610612745,...,None,NaN,0,None,None,None,None,1976-10-21 19:00:00,1976-10-21,19:00:00
51706,51706,61191,27600007,1976-10-21 19:00:00,New York,Knicks,1610612752,Los Angeles,Lakers,1610612747,...,None,NaN,0,None,None,None,None,1976-10-21 19:00:00,1976-10-21,19:00:00


In [10]:
print(transaction_timeframe["gameType"].unique())

['Regular Season' 'Preseason' 'Playoffs']


In [11]:
# Keep regular-season games for record reconstruction.
regular_season = transaction_timeframe[transaction_timeframe["gameType"] == "Regular Season"].copy()

In [12]:
print(regular_season["gameType"].unique())

['Regular Season']


In [13]:
regular_season["game_Date"] = pd.to_datetime(regular_season["game_Date"])

In [14]:
regular_season["game_Date"].dtype

dtype('<M8[ns]')

In [15]:
# Assign each game to its NBA season.
regular_season = regular_season.copy()

regular_season["season_start_year"] = regular_season["game_Date"].dt.year - (regular_season["game_Date"].dt.month < 7).astype(int)

regular_season["Season"] = (
    regular_season["season_start_year"].astype(str) + "-" + (regular_season["season_start_year"] + 1).astype(str).str[-2:]
)

In [16]:
# Capture the observed start and end date of each season.
season_boundaries = (
    regular_season.groupby("Season", as_index=False)
    .agg(season_start_year=("season_start_year", "first"), season_start_date=("game_Date", "min"), season_end_date=("game_Date", "max"))
    .sort_values("season_start_year")
    .reset_index(drop=True)
)

In [17]:
regular_season[["game_Date", "Season"]].head()

,game_Date,Season
0,2019-04-10,2018-19
1,2019-04-10,2018-19
2,2019-04-10,2018-19
3,2019-04-10,2018-19
4,2019-04-10,2018-19


In [18]:
regular_season[["game_Date", "Season"]].tail()

,game_Date,Season
51702,1976-10-22,1976-77
51703,1976-10-21,1976-77
51704,1976-10-21,1976-77
51705,1976-10-21,1976-77
51706,1976-10-21,1976-77


In [19]:
seasons = regular_season["Season"].unique()

In [20]:
# Count wins by team and season.
team_wins = regular_season.groupby(["Season", "winner"]).size().reset_index(name="Wins").rename(columns={"winner": "teamID"})

In [21]:
# Derive the losing team from the recorded winner.
regular_season["loser"] = np.where(
    regular_season["winner"] == regular_season["hometeamId"], regular_season["awayteamId"], regular_season["hometeamId"]
)

In [22]:
# Count losses by team and season.
team_losses = regular_season.groupby(["Season", "loser"]).size().reset_index(name="Losses").rename(columns={"loser": "teamID"})

In [23]:
# Build a season-aware team name lookup.
team_lookup = (
    regular_season[["Season", "hometeamId", "hometeamName"]]
    .rename(columns={"hometeamId": "teamID", "hometeamName": "teamName"})
    .drop_duplicates(subset=["Season", "teamID"])
    .reset_index(drop=True)
)

In [24]:
team_wins = team_wins.merge(team_lookup, on=["Season", "teamID"], how="left", validate="one_to_one")

In [25]:
# Combine wins and losses into complete season records.
team_records = team_wins.merge(team_losses, on=["Season", "teamID"], how="outer", validate="one_to_one")

In [26]:
team_records["Games"] = team_records["Wins"] + team_records["Losses"]

In [27]:
team_records = team_records.merge(team_lookup, on=["Season", "teamID", "teamName"], how="left", validate="one_to_one")

In [28]:
team_records = team_records.merge(season_boundaries, on="Season", how="left", validate="many_to_one")

In [29]:
team_records.columns.tolist()

['Season',
 'teamID',
 'Wins',
 'teamName',
 'Losses',
 'Games',
 'season_start_year',
 'season_start_date',
 'season_end_date']

In [30]:
schedule_summary = team_records.groupby(["Season", "Games"]).size().reset_index(name="Team_Count").sort_values(["Season", "Games"])

display(schedule_summary)

,Season,Games,Team_Count
0,1976-77,82,22
1,1977-78,82,22
2,1978-79,82,22
3,1979-80,82,22
4,1980-81,82,23
5,1981-82,82,23
6,1982-83,82,23
7,1983-84,82,23
8,1984-85,82,23
9,1985-86,82,23


In [31]:
# Save the team-season record table.
team_records.to_csv("../data/interim/team_season_records.csv")

In [32]:
team_records.head()

,Season,teamID,Wins,teamName,Losses,Games,season_start_year,season_start_date,season_end_date
0,1976-77,1610612737,31,Hawks,51,82,1976,1976-10-21,1977-04-10
1,1976-77,1610612738,44,Celtics,38,82,1976,1976-10-21,1977-04-10
2,1976-77,1610612739,43,Cavaliers,39,82,1976,1976-10-21,1977-04-10
3,1976-77,1610612741,44,Bulls,38,82,1976,1976-10-21,1977-04-10
4,1976-77,1610612743,50,Nuggets,32,82,1976,1976-10-21,1977-04-10


In [33]:
# Save the filtered regular-season game log.
regular_season.to_csv("../data/interim/all_regular_season_games.csv")